In [1]:
import pandas as pd
train_df = pd.read_parquet(r"final_data\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"final_data\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"final_data\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")

In [2]:
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')


Tạo sliding window cho LSTM/CNN

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split

TIME_STEPS = 10 # lấy 10 mẫu liên tiếp để dự đoán nhãn của mẫu cuối
BATCH_SIZE = 256 
NUM_CLASSES = len(train_df['label'].unique()) # số class
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {DEVICE}")


# Đĩnh nghĩa cửa số trượt
class SlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps):
        # Tách features và label, chuyển sang tensor
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        self.time_steps = time_steps
        
    def __len__(self):
        # Tổng số cửa sổ trượt tạo tạo ra từ dữ liệu
        return len(self.X) - self.time_steps + 1
        
    def __getitem__(self, idx):
        # Lấy 1 cửa sổ (từ idx đến idx + time_steps)    
        window_X = self.X[idx : idx + self.time_steps]
        # nhãn của cửa sổ đó là nhãn của mẫu cuối cùng trong cửa sổ
        label_y = self.y[idx + self.time_steps - 1]
        
        return window_X, label_y

print("Đang khởi tạo Dataset và DataLoader...")

# tạo datast cho train, val, test bằng SlidingWindowDataset
train_dataset = SlidingWindowDataset(train_df, TIME_STEPS)
val_dataset = SlidingWindowDataset(valid_df, TIME_STEPS)
test_dataset = SlidingWindowDataset(test_df, TIME_STEPS)

# Xóa Dataframe
import gc
del train_df, valid_df, test_df
gc.collect()

# 3. Tạo DataLoader
# Tham số shuffle=True ở train_loader sẽ tự động xáo trộn thứ tự các cửa sổ trượt
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

NUM_FEATURES = train_dataset.X.shape[1]
print(f"Số lượng features: {NUM_FEATURES}")

Đang sử dụng thiết bị: cuda
Đang khởi tạo Dataset và DataLoader...
Số lượng features: 314


5. Xay dựng mô hình CNN-BiLSTM bằng pytorch

In [4]:
class CNN_BiLSTM(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM, self).__init__()
        
        # tầng cnn iups lọc các đặc trưng quan trọng từ chuỗi thời gian
        # có 64 bộ lọc, mỗi bộ lọc có size = 3
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=64, kernel_size=3)
        
        # batch norm và activation
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        
        # Đầu vào LSTM có size 64
        # đi qua lớp BiLSTM với số LSTM units là 128
        self.bilstm = nn.LSTM(input_size=64, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        
        # BiLSTM có output size là hidden_size * 2 (vì bidirectional)
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # lớp fully connected cuối cùng để dự đoán nhãn
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x đang có shape: (Batch, TimeSteps, Features)
        
        # đảo trục cho Conv1d -> (Batch, Features, TimeSteps)
        x = x.permute(0, 2, 1) 
        
        # đi qua 1d-cnn
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.pool1(x)
        
        # đảo trục lại cho LSTM -> (Batch, NewTimeSteps, Channels)
        x = x.permute(0, 2, 1)
        
        # đi qua BiLSTM
        out, (hn, cn) = self.bilstm(x)
        
        # Lấy output của TimeStep cuối cùng để dự đoán
        # out shape: (Batch, SeqLen, Hidden*2) -> Lấy [:, -1, :]
        out = out[:, -1, :]
        
        # đi qua lớp dense để ra dự đoán nhãn
        out = self.dropout(out)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

# Khởi tạo mô hình và đẩy lên GPU
model = CNN_BiLSTM(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
print(model)

CNN_BiLSTM(
  (conv1): Conv1d(314, 64, kernel_size=(3,), stride=(1,))
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (bilstm): LSTM(64, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=256, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=11, bias=True)
)


6. Training mô hình (class_weights + sliding window)

In [5]:
# tính class weights và sqrt để tránh quá lệch trọng số
train_labels = train_dataset.y.numpy()
unique, counts = np.unique(train_labels, return_counts=True)
class_count_dict = dict(zip(unique, counts))

num_classes = len(class_count_dict)
counts = np.array([class_count_dict[i] for i in range(num_classes)])
total_samples = np.sum(counts)
class_weights = total_samples / (num_classes * counts)
smoothed_weights = np.sqrt(class_weights)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights_tensor = torch.tensor(smoothed_weights, dtype=torch.float32).to(device)

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# lớp tính hàm focal loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha # hệ số trọng số cho từng class, có thể truyền smoothed_weights vào đây
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# dùng focal loss với class weights đã tính được
criterion = FocalLoss(alpha=class_weights_tensor, gamma=2.0)

In [7]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

# khai báo loss và optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

# giảm learning rate nếu loss không cải thiện
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 5

for epoch in range(EPOCHS):
    # train model
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs) 
        loss = criterion(outputs, labels) # Tính Loss
        
        loss.backward() # Lan truyền ngược
        optimizer.step() # Cập nhật trọng số
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    # tính validation loss và macro F1
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    # kích hoạt Learning Rate Schedule
    scheduler.step(avg_val_loss)
    
    # Early Stopping dựa trên val loss
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        # Lưu lại tệp trọng số tốt nhất
        torch.save(model.state_dict(), "best_cnn_bilstm.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

# test trên tập test và in classification report
print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("best_cnn_bilstm.pth"))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 0.0452, Train Macro F1: 0.9192 | Val Loss: 0.2690, Val Macro F1: 0.8625


Epoch [2/30] - Train Loss: 0.0211, Train Macro F1: 0.9592 | Val Loss: 0.3007, Val Macro F1: 0.9205


Epoch [3/30] - Train Loss: 0.0172, Train Macro F1: 0.9690 | Val Loss: 0.2840, Val Macro F1: 0.9288


Epoch [4/30] - Train Loss: 0.0157, Train Macro F1: 0.9729 | Val Loss: 0.3457, Val Macro F1: 0.9177


Epoch [5/30] - Train Loss: 0.0115, Train Macro F1: 0.9828 | Val Loss: 0.3712, Val Macro F1: 0.9309


C:\Users\Admin\AppData\Local\Temp\ipykernel_2232\2464395618.py:93: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_cnn_bilstm.pth"))


Epoch [6/30] - Train Loss: 0.0109, Train Macro F1: 0.9836 | Val Loss: 0.3787, Val Macro F1: 0.9299

[Early Stopping] Model không cải thiện sau 5 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0       0.68      0.52      0.59     19837
           1       1.00      1.00      1.00    484077
           2       0.41      1.00      0.58      2515
           3       1.00      0.87      0.93     36253
           4       0.41      0.44      0.43     18979
           5       0.97      0.98      0.98      2451
           6       0.77      0.73      0.75     11847
           7       1.00      1.00      1.00    107436
           8       0.48      1.00      0.65     16746
           9       1.00      0.67      0.80     41523
          10       0.99      0.99      0.99     18567

    accuracy                           0.94    760231
   macro avg       0.79      0.84      0.79    760231
weighted avg       0.96      0.94      0.95    760231



In [8]:
print(classification_report(all_test_targets, all_test_preds))

              precision    recall  f1-score   support

           0       0.68      0.52      0.59     19837
           1       1.00      1.00      1.00    484077
           2       0.41      1.00      0.58      2515
           3       1.00      0.87      0.93     36253
           4       0.41      0.44      0.43     18979
           5       0.97      0.98      0.98      2451
           6       0.77      0.73      0.75     11847
           7       1.00      1.00      1.00    107436
           8       0.48      1.00      0.65     16746
           9       1.00      0.67      0.80     41523
          10       0.99      0.99      0.99     18567

    accuracy                           0.94    760231
   macro avg       0.79      0.84      0.79    760231
weighted avg       0.96      0.94      0.95    760231



7. Undersampling + Sliding Window

In [9]:
import pandas as pd
train_df = pd.read_parquet(r"final_data\chunk-based-split\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"final_data\chunk-based-split\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"final_data\chunk-based-split\test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [10]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000):
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        self.time_steps = time_steps
        
        print("Đang tính toán các cửa sổ hợp lệ và Undersampling...")
        
        # lấy số lượng cửa sổ có thể tạo ra từ dữ liệu
        num_possible_windows = len(self.X) - self.time_steps + 1
        # lấy nhãn của tất cả các cửa sổ có thể tạo ra  
        window_labels = self.y[self.time_steps - 1 : ].numpy()
        
        # tạo mảng index cho tất cả các cửa sổ có thể tạo ra
        all_indices = np.arange(num_possible_windows)
        
        self.valid_indices = []
        
        # lặp qua từng class
        classes, counts = np.unique(window_labels, return_counts=True)
        
        
        for c, count in zip(classes, counts):
            # lấy tất cả index của các cửa sổ thuộc class c
            c_indices = all_indices[window_labels == c]
            
            # nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # chọn ngẫu nhiên không hoàn lại max_samples_per_class index từ c_indices
                c_indices = np.random.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} cửa sổ.")
            else:
                # nếu ít hơn thì giữ nguyên
                print(f"Class {c}: Giữ nguyên {count} cửa sổ.")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn
        np.random.shuffle(self.valid_indices)
        print(f"Tổng số cửa sổ sau khi Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # lấy các index đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # lấy feature và label của cửa sổ trượt tại index thực sự
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        return window_X, label_y

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
MAX_SAMPLES = 20000 

print("Khởi tạo tập Train (có Undersampling)...")
train_dataset = UndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES)

print("Khởi tạo tập Val (Không Undersampling, giữ nguyên gốc)...")
# set max_samples_per_class cực lớn (vd 10 triệu) để không xóa mẫu nào
val_dataset = UndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000)
test_dataset = UndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling)...
Đang tính toán các cửa sổ hợp lệ và Undersampling...
Class 0: Giảm từ 64474 xuống 20000 cửa sổ.
Class 1: Giảm từ 1573223 xuống 20000 cửa sổ.
Class 2: Giữ nguyên 8162 cửa sổ.
Class 3: Giảm từ 117811 xuống 20000 cửa sổ.
Class 4: Giảm từ 61663 xuống 20000 cửa sổ.
Class 5: Giữ nguyên 7952 cửa sổ.
Class 6: Giảm từ 38485 xuống 20000 cửa sổ.
Class 7: Giảm từ 349151 xuống 20000 cửa sổ.
Class 8: Giảm từ 54413 xuống 20000 cửa sổ.
Class 9: Giảm từ 134941 xuống 20000 cửa sổ.
Class 10: Giảm từ 60323 xuống 20000 cửa sổ.
Tổng số cửa sổ sau khi Undersampling: 196114
Khởi tạo tập Val (Không Undersampling, giữ nguyên gốc)...
Đang tính toán các cửa sổ hợp lệ và Undersampling...
Class 0: Giữ nguyên 14871 cửa sổ.
Class 1: Giữ nguyên 363050 cửa sổ.
Class 2: Giữ nguyên 1880 cửa sổ.
Class 3: Giữ nguyên 27181 cửa sổ.
Class 4: Giữ nguyên 14230 cửa sổ.
Class 5: Giữ nguyên 1831 cửa sổ.
Class 6: Giữ nguyên 8881 cửa sổ.
Class 7: Giữ nguyên 80570 cửa sổ.
Class 8: Giữ nguyên 1

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_dataset.X.shape[1]

# model CNN-BiLSTM với Cơ chế Attention
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        # Lớp Linear để tính điểm số (score) cho từng time-step
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        # lstm_outputs shape: (Batch, SeqLen, Hidden*2)
        
        # 1. Tính điểm chú ý cho mỗi timestep -> Shape: (Batch, SeqLen, 1)
        scores = self.attention(lstm_outputs)
        
        # 2. Áp dụng Softmax dọc theo trục SeqLen để lấy trọng số (tổng các timestep = 1)
        weights = torch.softmax(scores, dim=1)
        
        # 3. Nhân trọng số với output của LSTM và tính tổng (Weighted Sum)
        # (Batch, SeqLen, 1) * (Batch, SeqLen, Hidden*2) -> (Batch, Hidden*2)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        
        return context_vector, weights

class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # --- Tầng 1D-CNN (Nâng cấp Sức chứa) ---
        # Block 1: Tăng từ num_features lên 128 (Thêm padding=1 để không mất chiều dài chuỗi quá nhanh)
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.dropout_cnn = nn.Dropout1d(0.2)

        # Block 2 (Mới): Lọc sâu thêm 1 lớp nữa, tăng kênh lên 256
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        
        self.relu = nn.ReLU()
        
        # Pooling gộp 2 bước lại làm 1 để giảm tải cho LSTM
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        # --- Tầng BiLSTM ---
        # Đầu vào của LSTM giờ là 256 (do out_channels của lớp Conv2)
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # tầng attention
        # Kích thước đầu vào của Attention = hidden_size * 2 (vì BiLSTM ghép 2 chiều ngược xuôi)
        self.attention = Attention(hidden_size * 2)
        
        self.dropout = nn.Dropout(0.5)
        
        # --- Tầng Phân loại (Dense) ---
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # Đầu vào x: (Batch, TimeSteps, Features)
        
        # 1. Đảo trục cho CNN -> (Batch, Features, TimeSteps)
        x = x.permute(0, 2, 1) 
        
        # Đi qua CNN Block 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout_cnn(x)

        # Đi qua CNN Block 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        
        # Pooling giảm chiều dài chuỗi
        x = self.pool(x)
        
        # 2. Đảo trục lại cho LSTM -> (Batch, NewTimeSteps, 256 Channels)
        x = x.permute(0, 2, 1)
        
        # Đi qua BiLSTM
        # out shape: (Batch, SeqLen, Hidden*2)
        out, (hn, cn) = self.bilstm(x)
        
        # 3. CƠ CHẾ ATTENTION (Thay vì lấy out[:, -1, :])
        # context_vector đã tổng hợp thông tin quan trọng nhất của toàn bộ cửa sổ
        context_vector, attn_weights = self.attention(out)
        
        # 4. Đi qua Dense
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

# Khởi tạo mô hình mới và đẩy lên thiết bị
model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
print(model)

CNN_BiLSTM_Attention(
  (conv1): Conv1d(314, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout_cnn): Dropout1d(p=0.2, inplace=False)
  (conv2): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (bilstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (attention): Attention(
    (attention): Linear(in_features=256, out_features=1, bias=True)
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=256, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=11, bias=True)
)


In [14]:
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight

# tính lại phân phối nhãn (labels) thực tế sau khi đã áp dụng Undersampling trong tập train
actual_labels = []
for idx in train_dataset.valid_indices:
    label = train_dataset.y[idx + TIME_STEPS - 1].item()
    actual_labels.append(label)

actual_labels = np.array(actual_labels)

classes_in_train = np.unique(actual_labels)
new_weights = compute_class_weight(
    class_weight='balanced', 
    classes=classes_in_train, 
    y=actual_labels
)

# căn bậc 2 trọng số
smoothed_new_weights = np.sqrt(new_weights)

# khởi tạo focal loss
alpha_tensor = torch.tensor(smoothed_new_weights, dtype=torch.float32).to(DEVICE)
criterion = FocalLoss(alpha=alpha_tensor, gamma=2.0)

print(f"Trọng số mới sau khi Undersampling và Smooth:\n{alpha_tensor.cpu().numpy()}")

Trọng số mới sau khi Undersampling và Smooth:
[0.94415426 0.94415426 1.4779497  0.94415426 0.94415426 1.4973377
 0.94415426 0.94415426 0.94415426 0.94415426 0.94415426]


In [15]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

# train với focal loss không trọng số
criterion = FocalLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 5

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_cnn_bilstm.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("best_cnn_bilstm.pth"))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/30] - Train Loss: 0.2410, Train Macro F1: 0.8560 | Val Loss: 0.0412, Val Macro F1: 0.9005


Epoch [2/30] - Train Loss: 0.0769, Train Macro F1: 0.9395 | Val Loss: 0.0502, Val Macro F1: 0.9085


Epoch [3/30] - Train Loss: 0.0611, Train Macro F1: 0.9498 | Val Loss: 0.0900, Val Macro F1: 0.8676


Epoch [4/30] - Train Loss: 0.0536, Train Macro F1: 0.9546 | Val Loss: 0.2011, Val Macro F1: 0.8382


...

Epoch [17/30] - Train Loss: 0.0205, Train Macro F1: 0.9829 | Val Loss: 0.0293, Val Macro F1: 0.9768


Epoch [18/30] - Train Loss: 0.0165, Train Macro F1: 0.9859 | Val Loss: 0.0318, Val Macro F1: 0.9776


C:\Users\Admin\AppData\Local\Temp\ipykernel_2232\3202232133.py:87: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_cnn_bilstm.pth"))


Epoch [19/30] - Train Loss: 0.0162, Train Macro F1: 0.9865 | Val Loss: 0.0342, Val Macro F1: 0.9616

[Early Stopping] Model không cải thiện sau 5 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0       0.93      0.92      0.93     19842
           1       1.00      1.00      1.00    484073
           2       0.98      1.00      0.99      2521
           3       1.00      0.99      0.99     36260
           4       0.95      0.89      0.92     18981
           5       0.95      1.00      0.97      2460
           6       0.94      0.91      0.92     11851
           7       1.00      1.00      1.00    107440
           8       0.82      0.98      0.89     16760
           9       0.99      0.97      0.98     41522
          10       0.98      0.98      0.98     18572

    accuracy                           0.99    760282
   macro avg       0.96      0.97      0.96    760282
weighted avg       0.99      0.99      0.99    760282



3. Undersampling + từng sample riêng lẻ (ko dùng sliding window) + CNN-BiLSTM-Attention (giống trong paper)

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_dataset.X.shape[1]
# ==========================================
# 1. Cơ chế Attention
# ==========================================
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        # Tính điểm chú ý
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        # Nhân trọng số và tính tổng
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# ==========================================
# 2. Kiến trúc chính bám sát Bài Báo
# ==========================================
class Paper_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Paper_CNN_BiLSTM_Attention, self).__init__()
        
        # --- Tầng CNN Block 1 ---
        # Bài báo: 128 filters, kernel size 5, ReLU [cite: 119] -> Batch Norm, MaxPool(2), Dropout(0.3) [cite: 120, 167]
        # in_channels = 1 (Vì ta coi 347 features như 1 chuỗi dài 347 đơn vị)
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=128, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout1d(0.3)

        # --- Tầng CNN Block 2 ---
        # Bài báo: 256 filters, kernel size 5, ReLU -> Batch Norm, MaxPool(2), Dropout(0.3) [cite: 121, 168, 169]
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout1d(0.3)

        # --- Tầng CNN Block 3 ---
        # Bài báo: 128 filters, kernel size 3, ReLU -> Batch Norm, MaxPool(2), Dropout(0.3) [cite: 121, 169, 170]
        self.conv3 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.drop3 = nn.Dropout1d(0.3)

        # --- Tầng BiLSTM ---
        # Bài báo: Bi-LSTM 128 units [cite: 122], theo sau là Dropout(0.3) [cite: 171]
        # Đầu vào của LSTM là 128 channels từ conv3
        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.drop_lstm = nn.Dropout(0.3)

        # --- Tầng Attention ---
        # Bài báo: Attention mechanism sau LSTM [cite: 122, 171]
        self.attention = Attention(128 * 2) # *2 vì là Bidirectional

        # --- Các Tầng Dense (Fully Connected) ---
        # Bài báo: Dense 256 + ReLU -> Dropout(0.4) [cite: 123, 171]
        self.fc1 = nn.Linear(128 * 2, 256)
        self.drop_fc1 = nn.Dropout(0.4)

        # Bài báo: Dense 128 + ReLU -> Dropout(0.4) [cite: 123, 172]
        self.fc2 = nn.Linear(256, 128)
        self.drop_fc2 = nn.Dropout(0.4)

        # Output layer: 10 class Softmax [cite: 124, 173] (Ta dùng num_classes của bạn = 11)
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Đầu vào x từ StandardFlowDataset đang có shape: (Batch, Features)
        
        # MẸO QUAN TRỌNG: Thêm chiều Channel = 1 để đưa vào Conv1d
        # Shape trở thành: (Batch, 1, Features)
        x = x.unsqueeze(1) 

        # --- Đi qua 3 Block CNN ---
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        # --- Chuẩn bị cho BiLSTM ---
        # Shape hiện tại: (Batch, Channels=128, NewLength)
        # BiLSTM cần shape: (Batch, SequenceLength, Features/Channels)
        x = x.permute(0, 2, 1)

        # --- Đi qua BiLSTM ---
        out, _ = self.bilstm(x)
        out = self.drop_lstm(out)

        # --- Đi qua Attention ---
        context_vector, _ = self.attention(out)

        # --- Đi qua Dense Layers ---
        out = self.fc1(context_vector)
        out = F.relu(out)
        out = self.drop_fc1(out)

        out = self.fc2(out)
        out = F.relu(out)
        out = self.drop_fc2(out)

        out = self.fc3(out)
        
        # Lưu ý: Không dùng F.softmax ở đây vì hàm nn.CrossEntropyLoss() đã tự động tính Softmax bên trong nó.
        return out

# Khởi tạo mô hình và đẩy lên GPU
model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)
print(model)

Paper_CNN_BiLSTM_Attention(
  (conv1): Conv1d(1, 128, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop1): Dropout1d(p=0.3, inplace=False)
  (conv2): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop2): Dropout1d(p=0.3, inplace=False)
  (conv3): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop3): Dropout1d(p=0.3, inplace=False)
  (bilstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (drop_lstm): Dropout(p=0.3, inp

In [17]:
from torch.utils.data import Dataset
class StandardFlowDataset(Dataset):
    def __init__(self, df, max_samples_per_class=None):
        self.X = []
        self.y = []
        
        # Nhóm theo từng class để Undersample
        for class_name, group in df.groupby('label'):
            if max_samples_per_class and len(group) > max_samples_per_class:
                group = group.sample(n=max_samples_per_class, random_state=42)
            
            self.X.append(group.drop(columns=['label']).values)
            self.y.append(group['label'].values)
            
        self.X = np.vstack(self.X)
        self.y = np.concatenate(self.y)
        
        # Xáo trộn dữ liệu
        idx = np.random.permutation(len(self.X))
        self.X = torch.tensor(self.X[idx], dtype=torch.float32)
        self.y = torch.tensor(self.y[idx], dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Khởi tạo lại
MAX_SAMPLES = 20000 
train_dataset = StandardFlowDataset(train_df, max_samples_per_class=MAX_SAMPLES)
val_dataset = StandardFlowDataset(valid_df) # Không truyền max_samples để giữ nguyên

In [18]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm import tqdm

# ==========================================
# 1. THIẾT LẬP HYPERPARAMETERS CHUẨN BÀI BÁO
# ==========================================
# Bài báo sử dụng hàm Loss chuẩn cho phân loại đa lớp
criterion = nn.CrossEntropyLoss()

# Bài báo cấu hình bộ tối ưu hóa Adam với learning rate là 0.001 
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Sử dụng Scheduler để giảm Learning Rate đi một nửa nếu Loss đi ngang
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

EPOCHS = 50
EARLY_STOP_PATIENCE = 10 # Cho phép mô hình kiên nhẫn hơn một chút
best_val_loss = float('inf')
patience_counter = 0

print("🚀 BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH 1D-CNN + BiLSTM + ATTENTION...")

# ==========================================
# 2. VÒNG LẶP HUẤN LUYỆN
# ==========================================
for epoch in range(EPOCHS):
    # --- TRAINING ---
    model.train()
    train_loss = 0.0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        # Đảm bảo dữ liệu nằm trên GPU
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        # Reset gradient
        optimizer.zero_grad()
        
        # Forward pass (Sinh dự đoán)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass (Tính toán đạo hàm)
        loss.backward()
        
        # 🛡️ BẢO HIỂM CHỐNG NỔ LOSS (Gradient Clipping)
        # Giới hạn độ lớn của Gradient không vượt quá 1.0, ngăn chặn Dying ReLU và NaN
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Cập nhật trọng số
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        loop.set_postfix(loss=loss.item())
        
    avg_train_loss = train_loss / len(train_loader.dataset)
    
    # --- VALIDATION ---
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    avg_val_loss = val_loss / len(val_loader.dataset)
    
    # Đánh giá bằng Macro-F1 (Công bằng cho mọi class)
    val_f1_macro = f1_score(all_targets, all_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro-F1: {val_f1_macro:.4f}")
    
    # Điều chỉnh Learning Rate dựa trên Validation Loss
    scheduler.step(avg_val_loss)
    
    # --- EARLY STOPPING & BÁO CÁO ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        
        # Lưu lại tệp trọng số xuất sắc nhất
        torch.save(model.state_dict(), "best_paper_model.pth")
        print(f" -> 🎉 Đã lưu model tốt nhất mới với Val Loss: {best_val_loss:.4f}\n")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n🛑 [Early Stopping] Mô hình không cải thiện Loss sau {EARLY_STOP_PATIENCE} epochs liên tiếp. Dừng huấn luyện.")
            break

print(f"🎯 Huấn luyện hoàn tất! Best Validation Loss đạt được: {best_val_loss:.4f}")

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


🚀 BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH 1D-CNN + BiLSTM + ATTENTION...


Epoch [1/50] - Train Loss: 1.0208 | Val Loss: 0.2664 | Val Macro-F1: 0.6594
 -> 🎉 Đã lưu model tốt nhất mới với Val Loss: 0.2664



Epoch [2/50] - Train Loss: 0.6304 | Val Loss: 0.1616 | Val Macro-F1: 0.7262
 -> 🎉 Đã lưu model tốt nhất mới với Val Loss: 0.1616



Epoch [3/50] - Train Loss: 0.5466 | Val Loss: 0.1800 | Val Macro-F1: 0.7081


...

Epoch [43/50] - Train Loss: 0.2523 | Val Loss: 0.0804 | Val Macro-F1: 0.8767


Epoch [44/50] - Train Loss: 0.2521 | Val Loss: 0.0774 | Val Macro-F1: 0.8798


Epoch [45/50] - Train Loss: 0.2519 | Val Loss: 0.0766 | Val Macro-F1: 0.8802


Epoch [46/50] - Train Loss: 0.2478 | Val Loss: 0.0776 | Val Macro-F1: 0.8937


Epoch [47/50] - Train Loss: 0.2461 | Val Loss: 0.0776 | Val Macro-F1: 0.8936

🛑 [Early Stopping] Mô hình không cải thiện Loss sau 10 epochs liên tiếp. Dừng huấn luyện.
🎯 Huấn luyện hoàn tất! Best Validation Loss đạt được: 0.0759


In [20]:
# báo cáo phân loại trên tập valid
print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP VALIDATION ---")
model.load_state_dict(torch.load("best_paper_model.pth"))
model.eval()
all_preds = []
all_targets = []
with torch.no_grad():
    for inputs, labels in tqdm(val_loader, desc="[Final Validation]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())
print(classification_report(all_targets, all_preds))

C:\Users\Admin\AppData\Local\Temp\ipykernel_2232\3024566931.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_paper_model.pth"))



--- BÁO CÁO PHÂN LOẠI TRÊN TẬP VALIDATION ---


[Final Validation]:   0%|          | 0/2228 [00:00<?, ?it/s]

              precision    recall  f1-score   support

           0       0.67      0.79      0.73     14880
           1       1.00      1.00      1.00    363050
           2       0.80      0.81      0.80      1880
           3       0.98      0.99      0.98     27181
           4       0.75      0.72      0.73     14230
           5       0.97      1.00      0.98      1831
           6       0.88      0.85      0.87      8881
           7       1.00      1.00      1.00     80570
           8       0.95      0.88      0.91     12551
           9       0.99      0.96      0.98     31140
          10       0.74      0.76      0.75     13920

    accuracy                           0.97    570114
   macro avg       0.89      0.89      0.89    570114
weighted avg       0.97      0.97      0.97    570114



In [21]:
# Báo cáo phân loại trên tập test
print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
test_dataset = StandardFlowDataset(test_df) # Tạo dataset test (giữ nguyên phân phối gốc)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
model.load_state_dict(torch.load("best_paper_model.pth"))
model.eval()
all_test_preds = []
all_test_targets = []
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())
print(classification_report(all_test_targets, all_test_preds))


--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


C:\Users\Admin\AppData\Local\Temp\ipykernel_2232\196052125.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_paper_model.pth"))


              precision    recall  f1-score   support

           0       0.68      0.80      0.73     19851
           1       1.00      1.00      1.00    484073
           2       0.76      0.82      0.79      2521
           3       0.98      0.98      0.98     36260
           4       0.76      0.73      0.74     18981
           5       0.99      1.00      1.00      2460
           6       0.90      0.81      0.85     11851
           7       1.00      1.00      1.00    107440
           8       0.95      0.89      0.92     16760
           9       0.99      0.97      0.98     41522
          10       0.75      0.77      0.76     18572

    accuracy                           0.97    760291
   macro avg       0.89      0.89      0.89    760291
weighted avg       0.97      0.97      0.97    760291



Do CNN mạnh về tính không gian, tuy nhiên bộ dataset này các cột không có tính không gian với nhau => CNN có thể không phát huy được sức mạnh => thay bằng lớp encoder khác

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

class Hybrid_DNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Hybrid_DNN_BiLSTM_Attention, self).__init__()
        
        # ==========================================
        # 1. KHỐI DNN (Thay thế 1D-CNN)
        # ==========================================
        self.dnn_extractor = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.1),
            
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.1),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.1)
        )

        # ==========================================
        # 2. KHỐI BiLSTM
        # ==========================================
        # Nhận 128 features từ khối DNN
        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.drop_lstm = nn.Dropout(0.1)

        # ==========================================
        # 3. KHỐI ATTENTION & OUTPUT
        # ==========================================
        self.attention = Attention(128 * 2) # *2 vì BiLSTM

        self.fc1 = nn.Linear(128 * 2, 128)
        self.drop_fc1 = nn.Dropout(0.1)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # x shape ban đầu: (Batch, 347)
        
        # Bước 1: Trích xuất đặc trưng bằng DNN
        features = self.dnn_extractor(x) # shape: (Batch, 128)
        
        # Bước 2: Thêm chiều Sequence Length = 1 để "lừa" LSTM
        # Shape trở thành: (Batch, 1, 128)
        lstm_input = features.unsqueeze(1) 
        
        # Bước 3: Đưa vào BiLSTM
        lstm_out, _ = self.bilstm(lstm_input) # lstm_out shape: (Batch, 1, 256)
        lstm_out = self.drop_lstm(lstm_out)
        
        # Bước 4: Đưa qua Attention
        context_vector, _ = self.attention(lstm_out) # context_vector shape: (Batch, 256)
        
        # Bước 5: Phân loại cuối cùng
        out = self.fc1(context_vector)
        out = F.leaky_relu(out, 0.1)
        out = self.drop_fc1(out)
        out = self.fc2(out)
        
        return out

# Khởi tạo mô hình
model = Hybrid_DNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

In [23]:
BATCH_SIZE = 1024
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm import tqdm

# ==========================================
# 1. THIẾT LẬP HYPERPARAMETERS CHUẨN BÀI BÁO
# ==========================================
# Bài báo sử dụng hàm Loss chuẩn cho phân loại đa lớp
criterion = nn.CrossEntropyLoss()

# Bài báo cấu hình bộ tối ưu hóa Adam với learning rate là 0.001 
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)

EPOCHS = 20
EARLY_STOP_PATIENCE = 10 # Cho phép mô hình kiên nhẫn hơn một chút
best_val_loss = float('inf')
patience_counter = 0

# Sử dụng Scheduler để suy giảm LR từ từ trong các epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=EPOCHS)

print("🚀 BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH DNN + BiLSTM + ATTENTION...")

# ==========================================
# 2. VÒNG LẶP HUẤN LUYỆN
# ==========================================
for epoch in range(EPOCHS):
    # --- TRAINING ---
    model.train()
    train_loss = 0.0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        # Đảm bảo dữ liệu nằm trên GPU
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        # Reset gradient
        optimizer.zero_grad()
        
        # Forward pass (Sinh dự đoán)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass (Tính toán đạo hàm)
        loss.backward()
        
        # 🛡️ BẢO HIỂM CHỐNG NỔ LOSS (Gradient Clipping)
        # Giới hạn độ lớn của Gradient không vượt quá 1.0, ngăn chặn Dying ReLU và NaN
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Cập nhật trọng số
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        loop.set_postfix(loss=loss.item())
        
    avg_train_loss = train_loss / len(train_loader.dataset)
    
    # --- VALIDATION ---
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    avg_val_loss = val_loss / len(val_loader.dataset)
    
    # Đánh giá bằng Macro-F1 (Công bằng cho mọi class)
    val_f1_macro = f1_score(all_targets, all_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro-F1: {val_f1_macro:.4f}")
    
    # CosineAnnealingLR step (không phụ thuộc vào chỉ số loss/metric nào, cứ tự động giảm)
    scheduler.step()
    
    # --- EARLY STOPPING & BÁO CÁO ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        
        # Lưu lại tệp trọng số xuất sắc nhất
        torch.save(model.state_dict(), "best_paper_model.pth")
        print(f" -> 🎉 Đã lưu model tốt nhất mới với Val Loss: {best_val_loss:.4f}\n")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n🛑 [Early Stopping] Mô hình không cải thiện Loss sau {EARLY_STOP_PATIENCE} epochs liên tiếp. Dừng huấn luyện.")
            break

print(f"🎯 Huấn luyện hoàn tất! Best Validation Loss đạt được: {best_val_loss:.4f}")

🚀 BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH DNN + BiLSTM + ATTENTION...


Epoch [1/20] - Train Loss: 1.6320 | Val Loss: 0.5629 | Val Macro-F1: 0.6474
 -> 🎉 Đã lưu model tốt nhất mới với Val Loss: 0.5629



Epoch [2/20] - Train Loss: 0.6742 | Val Loss: 0.1880 | Val Macro-F1: 0.7508
 -> 🎉 Đã lưu model tốt nhất mới với Val Loss: 0.1880



Epoch [3/20] - Train Loss: 0.5196 | Val Loss: 0.1571 | Val Macro-F1: 0.7785
 -> 🎉 Đã lưu model tốt nhất mới với Val Loss: 0.1571



Epoch [4/20] - Train Loss: 0.4802 | Val Loss: 0.1554 | Val Macro-F1: 0.7697
 -> 🎉 Đã lưu model tốt nhất mới với Val Loss: 0.1554



...

Epoch [16/20] - Train Loss: 0.3668 | Val Loss: 0.1068 | Val Macro-F1: 0.8330


Epoch [17/20] - Train Loss: 0.3637 | Val Loss: 0.1024 | Val Macro-F1: 0.8662
 -> 🎉 Đã lưu model tốt nhất mới với Val Loss: 0.1024



Epoch [18/20] - Train Loss: 0.3607 | Val Loss: 0.1020 | Val Macro-F1: 0.8674
 -> 🎉 Đã lưu model tốt nhất mới với Val Loss: 0.1020



Epoch [19/20] - Train Loss: 0.3596 | Val Loss: 0.1011 | Val Macro-F1: 0.8667
 -> 🎉 Đã lưu model tốt nhất mới với Val Loss: 0.1011



Epoch [20/20] - Train Loss: 0.3580 | Val Loss: 0.1014 | Val Macro-F1: 0.8668
🎯 Huấn luyện hoàn tất! Best Validation Loss đạt được: 0.1011


Chạy random forest để chọn ra top 100 cột có điểm cao nhất => đưa vào 1D-CNN để trích xuất đặc trưng => đưa vào BiLSTM + Attention để phân loại

In [25]:
import scipy.cluster.hierarchy as hc
from scipy.stats import spearmanr
import numpy as np

print("🔄 Đang phân tích sự tương quan của 347 đặc trưng...")
# Tính ma trận tương quan (Dùng Spearman cho an toàn với giá trị ngoại lai)
# Lưu ý: Chỉ tính trên tập Train (Dùng numpy array)
X_train_np = train_dataset.X.numpy()
corr = spearmanr(X_train_np).correlation

# Xử lý các giá trị NaN (nếu có) do cột toàn số 0
corr = np.nan_to_num(corr)

# Gom cụm phân cấp (Hierarchical Clustering)
print("🧩 Đang sắp xếp lại vị trí các cột để 1D-CNN học tốt nhất...")
corr_linkage = hc.ward(corr)
permuted_indices = hc.leaves_list(corr_linkage)

# Sắp xếp lại Dữ liệu theo trật tự không gian mới
train_dataset.X = train_dataset.X[:, permuted_indices]
val_dataset.X = val_dataset.X[:, permuted_indices]
test_dataset.X = test_dataset.X[:, permuted_indices]

print("✅ Đã sắp xếp xong! Các tính năng liên quan đã được xếp cạnh nhau.")

🔄 Đang phân tích sự tương quan của 347 đặc trưng...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


🧩 Đang sắp xếp lại vị trí các cột để 1D-CNN học tốt nhất...
✅ Đã sắp xếp xong! Các tính năng liên quan đã được xếp cạnh nhau.


In [26]:
BATCH_SIZE = 512
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm import tqdm

# Khai báo Hàm Loss và Optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

# Learning Rate Scheduler (Giảm LR nếu loss không cải thiện)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

EPOCHS = 35
best_val_loss = float('inf')
patience_counter = 0
EARLY_STOP_PATIENCE = 5

for epoch in range(EPOCHS):
    # --- TRAINING ---
    model.train()
    train_loss = 0.0
    total_train = 0

    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad() # Xóa gradient cũ
        
        outputs = model(inputs) # Forward
        loss = criterion(outputs, labels) # Tính Loss
        
        loss.backward() # Lan truyền ngược
        optimizer.step() # Cập nhật trọng số
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)

        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
    
    # --- VALIDATION ---
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    # Kích hoạt Learning Rate Scheduler
    scheduler.step(avg_val_loss)
    
    # Early Stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        # Lưu lại tệp trọng số tốt nhất
        torch.save(model.state_dict(), "best_cnn_bilstm.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện Loss sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

# --- IN BÁO CÁO CUỐI CÙNG ---
print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("best_cnn_bilstm.pth"))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/35] - Train Loss: 0.6699, Train Macro F1: 0.7452 | Val Loss: 0.2352, Val Macro F1: 0.7516


Epoch [2/35] - Train Loss: 0.4837, Train Macro F1: 0.8091 | Val Loss: 0.1381, Val Macro F1: 0.7695


Epoch [3/35] - Train Loss: 0.4492, Train Macro F1: 0.8240 | Val Loss: 0.1440, Val Macro F1: 0.8086


Epoch [4/35] - Train Loss: 0.4253, Train Macro F1: 0.8344 | Val Loss: 0.2201, Val Macro F1: 0.7548


...

Epoch [10/35] - Train Loss: 0.3224, Train Macro F1: 0.8769 | Val Loss: 0.1289, Val Macro F1: 0.8091


Epoch [11/35] - Train Loss: 0.3037, Train Macro F1: 0.8840 | Val Loss: 0.1654, Val Macro F1: 0.7544


C:\Users\Admin\AppData\Local\Temp\ipykernel_2232\3238904140.py:98: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_cnn_bilstm.pth"))


Epoch [12/35] - Train Loss: 0.3023, Train Macro F1: 0.8846 | Val Loss: 0.2256, Val Macro F1: 0.7792

[Early Stopping] Model không cải thiện Loss sau 5 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0       0.56      0.76      0.64     19851
           1       1.00      1.00      1.00    484073
           2       0.67      0.86      0.75      2521
           3       0.99      0.95      0.97     36260
           4       0.57      0.55      0.56     18981
           5       0.99      0.99      0.99      2460
           6       0.78      0.81      0.80     11851
           7       1.00      1.00      1.00    107440
           8       0.69      0.65      0.67     16760
           9       0.98      0.96      0.97     41522
          10       0.83      0.62      0.71     18572

    accuracy                           0.96    760291
   macro avg       0.82      0.83      0.82    760291
weighted avg       0.96      0.96      0.96    760291



SMOTE + Không cửa sổ trượt

In [28]:
y_train = train_df['label']
x_train = train_df.drop(columns=['label'])

In [29]:
y_train.value_counts()

label
1     1573223
7      349151
9      134941
3      117811
0       64483
4       61663
10      60323
8       54413
6       38485
2        8162
5        7952
Name: count, dtype: int64

In [30]:
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from imblearn.combine import SMOTETomek

In [ ]:
TARGET_COUNT = 100000
counts = y_train.value_counts().to_dict()



under_strategy = {label: TARGET_COUNT for label, count in counts.items() if count > TARGET_COUNT}
over_strategy = {label: TARGET_COUNT for label, count in counts.items() if count < TARGET_COUNT}

under = RandomUnderSampler(sampling_strategy=under_strategy, random_state=42)
smote_tomek = SMOTETomek(sampling_strategy=over_strategy, random_state=42, n_jobs=-1)

pipeline = Pipeline(steps=[('under', under), ('over', smote_tomek)])

print("Start to apply Hybrid Sampling (Under + Borderline SMOTE)...")
x_train_resampled, y_train_resampled = pipeline.fit_resample(x_train, y_train)

print(f"Number of samples after resampling: {len(y_train_resampled)}")
print(y_train_resampled.value_counts())

Start to apply Hybrid Sampling (Under + Borderline SMOTE)...


In [ ]:
# transform to dataloaders
train_resampled_df = pd.DataFrame(x_train_resampled, columns=x_train.columns)
train_resampled_df['label'] = y_train_resampled


C:\Users\Admin\AppData\Local\Temp\ipykernel_9000\3202516233.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_resampled_df['label'] = y_train_resampled


In [ ]:
# save train_resampled_df to csv
train_resampled_df.to_csv('train_resampled_with_SMOTETomek.csv', index=False)

In [ ]:
train_resampled_df = pd.read_csv('train_resampled_with_SMOTETomek.csv')

In [ ]:
from torch.utils.data import Dataset
import torch
import numpy as np
from torch.utils.data import DataLoader
class StandardFlowDataset(Dataset):
    def __init__(self, df, max_samples_per_class=None):
        self.X = []
        self.y = []
        
        # Nhóm theo từng class để Undersample
        for class_name, group in df.groupby('label'):
            if max_samples_per_class and len(group) > max_samples_per_class:
                group = group.sample(n=max_samples_per_class, random_state=42)
            
            self.X.append(group.drop(columns=['label']).values)
            self.y.append(group['label'].values)
            
        self.X = np.vstack(self.X)
        self.y = np.concatenate(self.y)
        
        # Xáo trộn dữ liệu
        idx = np.random.permutation(len(self.X))
        self.X = torch.tensor(self.X[idx], dtype=torch.float32)
        self.y = torch.tensor(self.y[idx], dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
BATCH_SIZE = 512
train_dataset = StandardFlowDataset(train_resampled_df)
val_dataset = StandardFlowDataset(valid_df)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_dataset = StandardFlowDataset(test_df)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_dataset.X.shape[1]
NUM_CLASSES = len(train_dataset.y.unique())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ==========================================
# 1. Cơ chế Attention
# ==========================================
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        # Tính điểm chú ý
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        # Nhân trọng số và tính tổng
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# ==========================================
# 2. Kiến trúc chính bám sát Bài Báo
# ==========================================
class Paper_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Paper_CNN_BiLSTM_Attention, self).__init__()
        
        # --- Tầng CNN Block 1 ---
        # Bài báo: 128 filters, kernel size 5, ReLU [cite: 119] -> Batch Norm, MaxPool(2), Dropout(0.3) [cite: 120, 167]
        # in_channels = 1 (Vì ta coi 347 features như 1 chuỗi dài 347 đơn vị)
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=128, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout1d(0.3)

        # --- Tầng CNN Block 2 ---
        # Bài báo: 256 filters, kernel size 5, ReLU -> Batch Norm, MaxPool(2), Dropout(0.3) [cite: 121, 168, 169]
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout1d(0.3)

        # --- Tầng CNN Block 3 ---
        # Bài báo: 128 filters, kernel size 3, ReLU -> Batch Norm, MaxPool(2), Dropout(0.3) [cite: 121, 169, 170]
        self.conv3 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.drop3 = nn.Dropout1d(0.3)

        # --- Tầng BiLSTM ---
        # Bài báo: Bi-LSTM 128 units [cite: 122], theo sau là Dropout(0.3) [cite: 171]
        # Đầu vào của LSTM là 128 channels từ conv3
        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.drop_lstm = nn.Dropout(0.3)

        # --- Tầng Attention ---
        # Bài báo: Attention mechanism sau LSTM [cite: 122, 171]
        self.attention = Attention(128 * 2) # *2 vì là Bidirectional

        # --- Các Tầng Dense (Fully Connected) ---
        # Bài báo: Dense 256 + ReLU -> Dropout(0.4) [cite: 123, 171]
        self.fc1 = nn.Linear(128 * 2, 256)
        self.drop_fc1 = nn.Dropout(0.4)

        # Bài báo: Dense 128 + ReLU -> Dropout(0.4) [cite: 123, 172]
        self.fc2 = nn.Linear(256, 128)
        self.drop_fc2 = nn.Dropout(0.4)

        # Output layer: 10 class Softmax [cite: 124, 173] (Ta dùng num_classes của bạn = 11)
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Đầu vào x từ StandardFlowDataset đang có shape: (Batch, Features)
        
        # MẸO QUAN TRỌNG: Thêm chiều Channel = 1 để đưa vào Conv1d
        # Shape trở thành: (Batch, 1, Features)
        x = x.unsqueeze(1) 

        # --- Đi qua 3 Block CNN ---
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        # --- Chuẩn bị cho BiLSTM ---
        # Shape hiện tại: (Batch, Channels=128, NewLength)
        # BiLSTM cần shape: (Batch, SequenceLength, Features/Channels)
        x = x.permute(0, 2, 1)

        # --- Đi qua BiLSTM ---
        out, _ = self.bilstm(x)
        out = self.drop_lstm(out)

        # --- Đi qua Attention ---
        context_vector, _ = self.attention(out)

        # --- Đi qua Dense Layers ---
        out = self.fc1(context_vector)
        out = F.relu(out)
        out = self.drop_fc1(out)

        out = self.fc2(out)
        out = F.relu(out)
        out = self.drop_fc2(out)

        out = self.fc3(out)
        
        # Lưu ý: Không dùng F.softmax ở đây vì hàm nn.CrossEntropyLoss() đã tự động tính Softmax bên trong nó.
        return out

# Khởi tạo mô hình và đẩy lên GPU
model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)
print(model)

Paper_CNN_BiLSTM_Attention(
  (conv1): Conv1d(1, 128, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop1): Dropout1d(p=0.3, inplace=False)
  (conv2): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop2): Dropout1d(p=0.3, inplace=False)
  (conv3): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop3): Dropout1d(p=0.3, inplace=False)
  (bilstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (drop_lstm): Dropout(p=0.3, inp

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm import tqdm

# ==========================================
# 1. THIẾT LẬP HYPERPARAMETERS CHUẨN BÀI BÁO
# ==========================================
# Bài báo sử dụng hàm Loss chuẩn cho phân loại đa lớp
criterion = nn.CrossEntropyLoss()

# Bài báo cấu hình bộ tối ưu hóa Adam với learning rate là 0.001 
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay= 1e-4)

# Sử dụng Scheduler để giảm Learning Rate đi một nửa nếu Loss đi ngang
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

EPOCHS = 50
EARLY_STOP_PATIENCE = 10 # Cho phép mô hình kiên nhẫn hơn một chút
best_val_loss = float('inf')
patience_counter = 0

print("🚀 BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH 1D-CNN + BiLSTM + ATTENTION...")

# ==========================================
# 2. VÒNG LẶP HUẤN LUYỆN
# ==========================================
for epoch in range(EPOCHS):
    # --- TRAINING ---
    model.train()
    train_loss = 0.0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        # Đảm bảo dữ liệu nằm trên GPU
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        # Reset gradient
        optimizer.zero_grad()
        
        # Forward pass (Sinh dự đoán)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass (Tính toán đạo hàm)
        loss.backward()
        
        # 🛡️ BẢO HIỂM CHỐNG NỔ LOSS (Gradient Clipping)
        # Giới hạn độ lớn của Gradient không vượt quá 1.0, ngăn chặn Dying ReLU và NaN
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Cập nhật trọng số
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        loop.set_postfix(loss=loss.item())
        
    avg_train_loss = train_loss / len(train_loader.dataset)
    
    # --- VALIDATION ---
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    avg_val_loss = val_loss / len(val_loader.dataset)
    
    # Đánh giá bằng Macro-F1 (Công bằng cho mọi class)
    val_f1_macro = f1_score(all_targets, all_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro-F1: {val_f1_macro:.4f}")
    
    # Điều chỉnh Learning Rate dựa trên Loss thay vì F1
    scheduler.step(avg_val_loss)
    
    # --- EARLY STOPPING & BÁO CÁO ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        
        # Lưu lại tệp trọng số xuất sắc nhất
        torch.save(model.state_dict(), "best_paper_model.pth")
        print(f" -> 🎉 Đã lưu model tốt nhất mới với Val Loss: {best_val_loss:.4f}\n")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n🛑 [Early Stopping] Mô hình không cải thiện Loss sau {EARLY_STOP_PATIENCE} epochs liên tiếp. Dừng huấn luyện.")
            break

print(f"🎯 Huấn luyện hoàn tất! Best Validation Loss đạt được: {best_val_loss:.4f}")

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


🚀 BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH 1D-CNN + BiLSTM + ATTENTION...


Epoch [1/50] - Train Loss: 0.5763 | Val Loss: 0.1025 | Val Macro-F1: 0.8039
 -> 🎉 Đã lưu model tốt nhất mới với Val Macro-F1: 0.8039



Epoch [2/50] - Train Loss: 0.3491 | Val Loss: 0.0980 | Val Macro-F1: 0.8041
 -> 🎉 Đã lưu model tốt nhất mới với Val Macro-F1: 0.8041



Epoch [3/50] - Train Loss: 0.3232 | Val Loss: 0.0995 | Val Macro-F1: 0.7636


...

Epoch [30/50] - Train Loss: 0.1829 | Val Loss: 0.0667 | Val Macro-F1: 0.8632


Epoch [31/50] - Train Loss: 0.1829 | Val Loss: 0.0730 | Val Macro-F1: 0.8609


Epoch [32/50] - Train Loss: 0.1821 | Val Loss: 0.0677 | Val Macro-F1: 0.8551


Epoch [33/50] - Train Loss: 0.1791 | Val Loss: 0.0683 | Val Macro-F1: 0.8624


Epoch [34/50] - Train Loss: 0.1783 | Val Loss: 0.0665 | Val Macro-F1: 0.8551

🛑 [Early Stopping] Mô hình không cải thiện F1 sau 10 epochs liên tiếp. Dừng huấn luyện.
🎯 Huấn luyện hoàn tất! Best Validation Macro-F1 đạt được: 0.8834


In [ ]:
# in ra báo cáo phân loại và ma trận nhầm lẫn trên tập Validation cho model tốt nhất
# Tải lại model tốt nhất
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm import tqdm
model.load_state_dict(torch.load("best_paper_model.pth"))
model.eval()
all_preds = []
all_targets = []
with torch.no_grad():
    for inputs, labels in tqdm(val_loader, desc="Evaluating Best Model on Val Set"):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())
print("\n📊 Classification Report:")
print(classification_report(all_targets, all_preds))
print("📉 Confusion Matrix:")
print(confusion_matrix(all_targets, all_preds))

C:\Users\Admin\AppData\Local\Temp\ipykernel_9000\4000774911.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_paper_model.pth"))
Eva


📊 Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.57      0.62     14882
           1       1.00      1.00      1.00    363052
           2       1.00      0.80      0.89      1884
           3       0.98      1.00      0.99     27187
           4       0.74      0.65      0.69     14231
           5       1.00      1.00      1.00      1836
           6       0.99      0.92      0.95      8883
           7       1.00      1.00      1.00     80574
           8       0.97      0.79      0.87     12558
           9       1.00      1.00      1.00     31140
          10       0.59      0.87      0.70     13922

    accuracy                           0.97    570149
   macro avg       0.90      0.87      0.88    570149
weighted avg       0.97      0.97      0.97    570149

📉 Confusion Matrix:
[[  8536      0      0      0   2245      1      1      0     14      0
    4085]
 [     0 363052      0      0      0      0      0      0  

In [ ]:
import torch
import numpy as np
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm
import optuna

# --- 1. HÀM ÁP DỤNG NGƯỠNG CHO CLASS 10 ---
def apply_stomp_threshold(y_probs, threshold_c10=0.5):
    """
    Hàm ép mô hình phải "tự tin" hơn khi dự đoán Class 10 (STOMP).
    """
    # Lấy ra Top 1 và Top 2 class có xác suất cao nhất
    top2_probs, top2_classes = torch.topk(y_probs, k=2, dim=1)
    
    top1_probs = top2_probs[:, 0]
    top1_classes = top2_classes[:, 0]
    top2_classes = top2_classes[:, 1]
    
    final_preds = top1_classes.clone()
    
    # Kìm hãm Class 10: Nếu đoán là 10 nhưng xác suất < threshold -> Đẩy về Top 2
    mask_c10 = (top1_classes == 10) & (top1_probs < threshold_c10)
    final_preds[mask_c10] = top2_classes[mask_c10]
    
    return final_preds.cpu().numpy()

# --- 2. VÒNG LẶP LẤY XÁC SUẤT TRÊN TẬP VALIDATION ---
# Giả sử bạn đã định nghĩa val_loader và model
all_targets = []
all_probs = []

model.eval()
with torch.no_grad():
    for inputs, labels in tqdm(val_loader, desc="Lấy xác suất trên Val Set"):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        
        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs.cpu())
        all_targets.extend(labels.cpu().numpy())

y_probs_val = torch.cat(all_probs, dim=0)
y_val_true = np.array(all_targets)

y_probs_np = y_probs_val.numpy()
print("\n🚀 BẮT ĐẦU TỐI ƯU HÓA TRỌNG SỐ TOÀN CỤC CHO 11 CLASS BẰNG OPTUNA...")

# 1. Định nghĩa hàm mục tiêu cho Optuna
def objective(trial):
    # Optuna sẽ tự động sinh ra 11 trọng số W trong khoảng từ 0.1 đến 3.0
    weights = [trial.suggest_float(f'w_{i}', 0.1, 3.0) for i in range(11)]
    weights = np.array(weights)
    
    # Nhân xác suất gốc với trọng số
    weighted_probs = y_probs_np * weights
    
    # Lấy class có xác suất mới cao nhất
    y_pred_tuned = np.argmax(weighted_probs, axis=1)
    
    # Trả về Macro F1 để Optuna tối đa hóa
    return f1_score(y_val_true, y_pred_tuned, average='macro')

# 2. Khởi chạy quá trình dò tìm
# Tắt log rác của Optuna cho đỡ rối mắt
optuna.logging.set_verbosity(optuna.logging.WARNING) 
study = optuna.create_study(direction='maximize')

# Chạy 300 vòng thử nghiệm (chỉ mất vài giây vì toàn là phép toán trên ma trận numpy)
study.optimize(objective, n_trials=300, show_progress_bar=True)

# 3. Lấy ra bộ trọng số chiến thắng
best_weights = np.array([study.best_params[f'w_{i}'] for i in range(11)])

print("="*60)
print(f"🎉 Đã chốt được bộ trọng số tối ưu!")
print(f"👉 Trọng số tốt nhất: {np.round(best_weights, 3)}")
print(f"🚀 Macro F1 trên tập Valid đạt: {study.best_value:.4f}")
print("="*60)

# --- 4. IN BÁO CÁO TRÊN TẬP VALID ĐỂ KIỂM CHỨNG ---
weighted_probs_val = y_probs_np * best_weights
final_val_preds = np.argmax(weighted_probs_val, axis=1)

print("\n📊 BÁO CÁO PHÂN LOẠI TRÊN TẬP VALIDATION (SAU KHI ÁP DỤNG TRỌNG SỐ):")
print(classification_report(y_val_true, final_val_preds, digits=4))

Lấy xác suất trên Val Set: 100%|██████████| 1188/1188 [00:15<00:00, 75.11it/s]



🚀 BẮT ĐẦU TỐI ƯU HÓA TRỌNG SỐ TOÀN CỤC CHO 11 CLASS BẰNG OPTUNA...


  0%|          | 0/300 [00:00<?, ?it/s]

🎉 Đã chốt được bộ trọng số tối ưu!
👉 Trọng số tốt nhất: [1.01  2.461 2.645 2.672 0.507 0.966 2.24  1.354 2.204 1.265 0.394]
🚀 Macro F1 trên tập Valid đạt: 0.9102

📊 BÁO CÁO PHÂN LOẠI TRÊN TẬP VALIDATION (SAU KHI ÁP DỤNG TRỌNG SỐ):
              precision    recall  f1-score   support

           0     0.6537    0.8678    0.7457     15875
           1     1.0000    0.9998    0.9999    387256
           2     0.9680    0.9483    0.9580      2010
           3     0.9854    0.9997    0.9925     29001
           4     0.8709    0.6271    0.7292     15180
           5     0.9175    0.9821    0.9487      1959
           6     0.9667    0.8927    0.9282      9475
           7     1.0000    0.9982    0.9991     85946
           8     0.8951    0.9204    0.9076     13396
           9     1.0000    1.0000    1.0000     33217
          10     0.8295    0.7797    0.8038     14851

    accuracy                         0.9778    608166
   macro avg     0.9170    0.9105    0.9102    608166
weighted av

Hybrid sampling + XGBoost

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import f1_score

In [ ]:
TARGET_COL = "label"
x_train = train_resampled_df.drop(columns=[TARGET_COL])
y_train = train_resampled_df[TARGET_COL]

x_val = valid_df.drop(columns=[TARGET_COL])
y_val = valid_df[TARGET_COL]


In [ ]:
def custom_macro_f1(y_true, y_pred):
    # Kiểm tra số chiều của mảng dự đoán
    if y_pred.ndim == 2:
        # Nếu là mảng 2D (xác suất), lấy nhãn có xác suất cao nhất
        y_pred_labels = np.argmax(y_pred, axis=1)
    else:
        # Nếu đã là mảng 1D (nhãn trực tiếp), giữ nguyên
        y_pred_labels = y_pred 
        
    # Tính toán Macro F1
    f1 = f1_score(y_true, y_pred_labels, average='macro')
    
    return f1

In [ ]:
early_stop = xgb.callback.EarlyStopping(
    rounds = 100, # Dừng nếu không cải thiện sau 50 vòng
    metric_name = "custom_macro_f1",
    maximize = True,
    save_best = True
)
xgb_model = xgb.XGBClassifier(
    objective = "multi:softprob",
    num_class = NUM_CLASSES,
    n_estimators = 2000,
    learning_rate = 0.05,
    max_depth = 6,
    subsample = 0.7,
    colsample_bytree = 0.8,
    tree_method = "hist",
    device = "cuda",
    random_state = 42,
    eval_metric = custom_macro_f1,
    callbacks = [early_stop]
)



xgb_model.fit(
    x_train, y_train,
    eval_set = [(x_train, y_train), (x_val, y_val)],
    verbose = 10
)

model_path = "xgb_best_model.json"
xgb_model.save_model(model_path)

print(f"✅ Đã lưu model XGBoost tốt nhất vào: {model_path}")
print(f"Best Validation Macro-F1: {xgb_model.best_score:.4f}")

[0]	validation_0-mlogloss:2.16325	validation_0-custom_macro_f1:0.91767	validation_1-mlogloss:2.13239	validation_1-custom_macro_f1:0.87956
[10]	validation_0-mlogloss:1.15181	validation_0-custom_macro_f1:0.92395	validation_1-mlogloss:1.05014	validation_1-custom_macro_f1:0.88073
[20]	validation_0-mlogloss:0.74353	validation_0-custom_macro_f1:0.92880	validation_1-mlogloss:0.62654	validation_1-custom_macro_f1:0.88885
[30]	validation_0-mlogloss:0.51911	validation_0-custom_macro_f1:0.93003	validation_1-mlogloss:0.39805	validation_1-custom_macro_f1:0.89127
[40]	validation_0-mlogloss:0.38652	validation_0-custom_macro_f1:0.93152	validation_1-mlogloss:0.26547	validation_1-custom_macro_f1:0.89277
[50]	validation_0-mlogloss:0.30390	validation_0-custom_macro_f1:0.93216	validation_1-mlogloss:0.18575	validation_1-custom_macro_f1:0.89327
[60]	validation_0-mlogloss:0.25190	validation_0-custom_macro_f1:0.93318	validation_1-mlogloss:0.13731	validation_1-custom_macro_f1:0.89372
[70]	validation_0-mlogloss:0

In [ ]:
# Tải lại model XGBoost tốt nhất và đánh giá trên tập Validation
best_xgb_model = xgb.XGBClassifier()
model_path = "xgb_best_model.json"
best_xgb_model.load_model(model_path)
y_val_pred = best_xgb_model.predict(x_val)
val_f1_macro = f1_score(y_val, y_val_pred, average='macro')
print(f"Validation Macro-F1 của model XGBoost tốt nhất: {val_f1_macro:.4f}")

Validation Macro-F1 của model XGBoost tốt nhất: 0.9224


In [ ]:
# test f1 score trên tập test
x_test = test_df.drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL]
y_test_pred = best_xgb_model.predict(x_test)
test_f1_macro = f1_score(y_test, y_test_pred, average='macro')
print(f"Test Macro-F1 của model XGBoost tốt nhất: {test_f1_macro:.4f}")

(570154, 314)


XGBoost thông thường

In [ ]:
import pandas as pd
train_df = pd.read_parquet(r"final_data\chunk-based-split\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"final_data\chunk-based-split\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"final_data\chunk-based-split\test_df_prepared.parquet", engine="pyarrow")

In [ ]:
cols_to_drop = ["flow_id", "timestamp", "src_ip", "src_port", "dst_ip", "dst_port"]
train_df = train_df.drop(columns=cols_to_drop)
valid_df = valid_df.drop(columns=cols_to_drop)
test_df = test_df.drop(columns=cols_to_drop)

In [ ]:
import xgboost as xgb
from sklearn.metrics import log_loss, f1_score

# Tách Features và Label
X_train = train_df.drop(columns=['label'])
y_train = train_df['label']

X_valid = valid_df.drop(columns=['label'])
y_valid = valid_df['label']

# Khởi tạo model XGBoost
# SỬ DỤNG TRỰC TIẾP `early_stopping_rounds` ĐỂ AVOID ATTRIBUTE ERROR.
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    n_estimators=1000,          # Số lượng cây tối đa
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=50,   # SỬA LỖI: Cần khai báo số lượng vòng dừng sớm rõ ràng tại đây
    tree_method='hist',         # Sử dụng GPU nếu được hỗ trợ (cần device='cuda')
    device='cuda'               # Bật dòng này nếu bạn có GPU và đã cài xgboost-gpu
)

# Training Model
print("🚀 Bắt đầu quá trình huấn luyện XGBoost...")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    verbose=50                     # In ra tiến trình mỗi 50 epoch
)

# Lấy ra iteration tốt nhất (xác định bởi best validation score)
best_iteration = xgb_model.best_iteration
print(f"\n🎉 Mô hình tốt nhất đạt được tại vòng lặp thứ: {best_iteration}")

# Sinh dự đoán (Hàm predict() của XGBoost tự động lấy trọng số từ vòng lặp tốt nhất)
val_preds = xgb_model.predict(X_valid)
val_probs = xgb_model.predict_proba(X_valid)

# Đánh giá lại tự động mô hình tốt nhất
best_val_loss = log_loss(y_valid, val_probs)
best_val_f1_macro = f1_score(y_valid, val_preds, average='macro')

print(f"🎯 Đánh giá Best Model trên Valid Set | Val Loss: {best_val_loss:.4f} | Val Macro-F1: {best_val_f1_macro:.4f}")

# Lưu Model để sử dụng sau này
xgb_model.save_model("xgb_best_model.json")
print("💾 Đã lưu XGBoost Classifier tốt nhất tại 'xgb_best_model.json'")

In [ ]:
# đánh giá trên tập test
X_test = test_df.drop(columns=['label'])
y_test = test_df['label']
y_test_pred = xgb_model.predict(X_test)
test_f1_macro = f1_score(y_test, y_test_pred, average='macro')
print(f"Test Macro-F1 của model XGBoost tốt nhất: {test_f1_macro:.4f}")

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:751: UserWarning: [12:49:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Test Macro-F1 của model XGBoost tốt nhất: 0.9344
